In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_pickle("../outputs/cleaned_data.pkl")

In [ ]:
df.head()

## Top 10 pays par chiffre d'affaires

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
country_revenue

In [ ]:
plt.close('all')

In [ ]:
#Graphique
plt.figure(figsize=(10,5))

plt.bar(country_revenue.index, country_revenue.values)

plt.title("Top 10 Countries by Revenue")
plt.ylabel("Revenue")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("../visuals/top_10_countries.png", bbox_inches='tight')

plt.show()

In [ ]:
country_revenue.to_csv("../outputs/country_revenue.csv")

## Top produits

In [ ]:
top_products = (
    df.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

In [ ]:
plt.figure(figsize=(10,5))
plt.barh(top_products.index, top_products.values)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")

plt.tight_layout()

plt.savefig("../visuals/top_10_products.png", bbox_inches='tight')

plt.show()

## Top clients

In [ ]:
top_customers = (
    df.groupby('CustomerID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_customers

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(top_customers.index.astype(str), top_customers.values)

plt.title("Top 10 Customers by Revenue")
plt.ylabel("Revenue")

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("../visuals/top_10_customers.png", bbox_inches='tight')

plt.show()

## Analyse RFM

In [ ]:
# Date de référence
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
# Calcul RFM
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

In [ ]:
def safe_qcut(series, q):
    return pd.qcut(series, q=q, labels=False, duplicates='drop') + 1

rfm['R_score'] = safe_qcut(rfm['Recency'], 4)
rfm['F_score'] = safe_qcut(rfm['Frequency'], 4)
rfm['M_score'] = safe_qcut(rfm['Monetary'], 4)

In [ ]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [ ]:
# Segmentation
def segment(row):
    if row['RFM_Score'] == '444':
        return 'Champions'
    elif row['R_score'] == 4:
        return 'Loyal'
    elif row['F_score'] == 4:
        return 'Frequent'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)

In [ ]:
# Distribution des segments
segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(8,5))
plt.bar(segment_counts.index, segment_counts.values)

plt.title("Customer Segments Distribution")
plt.ylabel("Number of Customers")

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("../visuals/top_10_customerSD.png", bbox_inches='tight')

plt.show()

## Evolution du chiffre d'affaires dans le temps

In [ ]:
df['Month'] = df['InvoiceDate'].dt.to_period('M')

monthly_revenue = df.groupby('Month')['Revenue'].sum()

monthly_revenue

In [ ]:
# Visualisation
plt.figure(figsize=(10,5))

monthly_revenue.plot()

plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue")

plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("../visuals/monthly_Revenue_Trend.png", bbox_inches='tight')

plt.show()

## Analyse par jours de la semaine

In [ ]:
df['Day'] = df['InvoiceDate'].dt.day_name()

daily_revenue = df.groupby('Day')['Revenue'].sum()

daily_revenue = daily_revenue.reindex([
    'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'
])

In [ ]:
# Graph
plt.figure(figsize=(8,5))
plt.bar(daily_revenue.index, daily_revenue.values)

plt.title("Revenue by Day of Week")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("../visuals/revenue_by_day_of_week.png", bbox_inches='tight')

plt.show()

In [ ]:
df.to_pickle("../outputs/cleaned_data.pkl")

In [ ]:
df